# Protein (RPPA) Preprocessing — TCGA-BRCA
Produces 8 experimental dataset variants in `statistical_filtered/`.

Notebook location: `Thesis_v3/01_Causal_feature_extraction/Proteins/`

Protein-specific notes vs RNA/CNV:
- RPPA data is already log2 ratio scale — no log2 transform needed
- Small proteome (~460 proteins after QC) — variance percentile thresholds
  produce similar absolute counts to RNA/CNV at the same percentile cutoffs
- No z-score normalization (RPPA values are already centered near 0)

| V | Name | Selection logic |
|---|---|---|
| 1 | ultra_conservative | Variance > 98.3 percentile |
| 2 | conservative | Variance > 97.5 percentile |
| 3 | standard | Variance > 95.9 percentile |
| 4 | fdr_significant | Spearman FDR < 0.05, top proteins by composite |
| 5 | balanced | Var top 25% then Spearman top 10% |
| 6 | correlation | Abs. Spearman > 97.5 percentile |
| 7 | top_correlated | Top 100 positive + top 100 negative Spearman |
| 8 | composite | Average rank of Spearman + MI + Distance corr |


In [2]:
from pathlib import Path

# Notebook: Thesis_v3/01_Causal_feature_extraction/Proteins/proteins_preprocessing.ipynb
_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "data" / "TCGA-BRCA.protein.tsv").exists()
)

PROTEIN_FILE = BASE_DIR / "data" / "TCGA-BRCA.protein.tsv"
OUTCOME_FILE = BASE_DIR / "data" / "outcome.csv"
OUTPUT_DIR   = BASE_DIR / "01_Causal_feature_extraction" / "Proteins" / "statistical_filtered"
STATS_CACHE  = BASE_DIR / "01_Causal_feature_extraction" / "Proteins" / "protein_statistics_cache.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_PROTEINS = 50
TARGET_BROAD = 200   # proteome is small (~460 proteins); 1000 is not feasible

assert PROTEIN_FILE.exists(), f"Not found: {PROTEIN_FILE}"
assert OUTCOME_FILE.exists(), f"Not found: {OUTCOME_FILE}"

print(f"Base   : {BASE_DIR}")
print(f"Output : {OUTPUT_DIR}")
print(f"Cache  : {STATS_CACHE}")
print(f"TARGET_BROAD = {TARGET_BROAD}  (proteome too small for 1000)")
print("Paths OK")


Base   : C:\Users\olegk\Desktop\Thesis_v3
Output : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Proteins\statistical_filtered
Cache  : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Proteins\protein_statistics_cache.csv
TARGET_BROAD = 200  (proteome too small for 1000)
Paths OK


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")


In [4]:
print("Loading protein data...")
prot_raw = pd.read_csv(PROTEIN_FILE, sep="\t", index_col=0)
print(f"  Raw shape (proteins x samples): {prot_raw.shape}")

# Transpose to samples x proteins
prot = prot_raw.T.copy()
print(f"  Transposed: {prot.shape}  (samples x proteins)")

vals = prot.values.flatten()
vals = vals[~np.isnan(vals)]
print(f"  Value range: [{vals.min():.3f}, {vals.max():.3f}]")
print(f"  Mean: {vals.mean():.4f}  Std: {vals.std():.4f}")
print(f"  RPPA log2 ratio — already centered near 0, no log2 transform needed")

print("Loading outcome...")
outcome = pd.read_csv(OUTCOME_FILE, index_col=0)
print(f"  Samples  : {len(outcome)}")
print(f"  OS.time  : {outcome['OS.time'].min():.0f} - {outcome['OS.time'].max():.0f} days")
print(f"  Events   : {int(outcome['OS'].sum())} ({outcome['OS'].mean()*100:.1f}%)")

common  = sorted(set(prot.index) & set(outcome.index))
prot    = prot.loc[common].copy()
outcome = outcome.loc[common].copy()
print(f"  Overlap  : {len(common)} samples")


Loading protein data...
  Raw shape (proteins x samples): (487, 919)
  Transposed: (919, 487)  (samples x proteins)
  Value range: [-6.365, 6.920]
  Mean: 0.0190  Std: 0.5553
  RPPA log2 ratio — already centered near 0, no log2 transform needed
Loading outcome...
  Samples  : 1076
  OS.time  : 1 - 8605 days
  Events   : 150 (13.9%)
  Overlap  : 859 samples


In [5]:
print("Quality control and imputation...")

# Missing data analysis
miss = prot.isnull().mean(axis=0)
print(f"  100% missing proteins : {(miss == 1.0).sum()}  (drop)")
print(f"   >20% missing proteins: {(miss  > 0.20).sum()}")
print(f"     0% missing proteins: {(miss == 0.00).sum()}")

# Drop completely missing proteins
prot = prot.loc[:, miss < 1.0]
miss = prot.isnull().mean(axis=0)

# Drop proteins with >20% missing
prot = prot.loc[:, miss <= 0.20]
print(f"  After dropping >20% missing: {prot.shape[1]} proteins")

# Median imputation for residual missing values
imp  = SimpleImputer(strategy="median")
prot = pd.DataFrame(imp.fit_transform(prot), index=prot.index, columns=prot.columns)
print(f"  After imputation: {prot.isnull().sum().sum()} missing values remain")
print(f"  Final shape: {prot.shape}  (samples x proteins)")


Quality control and imputation...
  100% missing proteins : 23  (drop)
   >20% missing proteins: 24
     0% missing proteins: 217
  After dropping >20% missing: 463 proteins
  After imputation: 0 missing values remain
  Final shape: (859, 463)  (samples x proteins)


In [6]:
if STATS_CACHE.exists():
    print(f"Loading cached statistics ({STATS_CACHE.name})...")
    stats = pd.read_csv(STATS_CACHE)
    print(f"  Loaded: {len(stats):,} proteins")
else:
    import time
    os_time = outcome["OS.time"].values
    proteins = prot.columns.tolist()
    n        = len(proteins)
    print(f"Computing statistics for {n:,} proteins...")

    # 1. Spearman correlation
    print("  [1/3] Spearman correlation...")
    t0 = time.time()
    rho_arr, pval_arr = [], []
    for p in proteins:
        r, pv = spearmanr(prot[p].values, os_time)
        rho_arr.append(0.0 if np.isnan(r) else float(r))
        pval_arr.append(1.0 if np.isnan(pv) else float(pv))
    _, fdr_arr, _, _ = multipletests(pval_arr, method="fdr_bh")
    print(f"    Done in {time.time()-t0:.1f}s")

    # 2. Mutual information
    print("  [2/3] Mutual information...")
    mi_arr = mutual_info_regression(
        prot.values, os_time, random_state=42, n_neighbors=5
    )

    # 3. Distance correlation (rank-based approximation)
    # Same approximation as RNA and CNV for cross-modality consistency.
    print("  [3/3] Distance correlation (rank approximation)...")
    os_rank = rankdata(os_time).astype(float)
    dc_arr  = []
    for p in proteins:
        x_rank = rankdata(prot[p].values).astype(float)
        dc_arr.append(abs(float(np.corrcoef(x_rank, os_rank)[0, 1])))

    stats = pd.DataFrame({
        "protein":       proteins,
        "spearman_rho":  rho_arr,
        "abs_spearman":  np.abs(rho_arr),
        "pval":          pval_arr,
        "fdr":           fdr_arr,
        "mutual_info":   mi_arr,
        "distance_corr": dc_arr,
        "variance":      prot.var().values,
    })

    stats["rank_spearman"] = rankdata(-stats["abs_spearman"])
    stats["rank_mi"]       = rankdata(-stats["mutual_info"])
    stats["rank_dcor"]     = rankdata(-stats["distance_corr"])
    stats["composite"]     = (
        stats["rank_spearman"] + stats["rank_mi"] + stats["rank_dcor"]
    ) / 3.0
    stats = stats.sort_values("composite").reset_index(drop=True)

    stats.to_csv(STATS_CACHE, index=False)
    print(f"  Cached to {STATS_CACHE.name}")

n_fdr = int((stats["fdr"] < 0.05).sum())
print(f"  FDR < 0.05  : {n_fdr:,} / {len(stats):,} proteins")
print(f"  Top 10 by composite score:")
print(stats[["protein","abs_spearman","mutual_info","distance_corr","composite"]].head(10).to_string(index=False))


Computing statistics for 463 proteins...
  [1/3] Spearman correlation...
    Done in 0.3s
  [2/3] Mutual information...
  [3/3] Distance correlation (rank approximation)...
  Cached to protein_statistics_cache.csv
  FDR < 0.05  : 115 / 463 proteins
  Top 10 by composite score:
       protein  abs_spearman  mutual_info  distance_corr  composite
    PRC1_pT481      0.147503     0.054506       0.147503   9.333333
 DNA-Ligase-IV      0.149399     0.034596       0.149399  15.000000
 P38_pT180Y182      0.165077     0.025454       0.165077  24.000000
TUBERIN_pT1462      0.160969     0.025056       0.160969  27.000000
 Hexokinase-II      0.122536     0.031300       0.122536  32.333333
          DVL3      0.111478     0.057060       0.111478  35.000000
      PAXILLIN      0.132186     0.027458       0.132186  35.000000
         SETD2      0.136373     0.023424       0.136373  37.333333
 4EBP1_pT37T46      0.125552     0.027201       0.125552  38.000000
      GSK3_pS9      0.144222     0.020295 

In [7]:
print("Creating 8 dataset variants...")
print(f"Output: {OUTPUT_DIR}")
print(f"Note: proteome is small ({len(stats)} proteins); TARGET_BROAD={TARGET_BROAD}")

def build(protein_list, min_p=MIN_PROTEINS):
    """Select proteins; pad with top composite proteins if below minimum.
    No normalization — RPPA values are already on log2 ratio scale."""
    protein_list = [p for p in protein_list if p in prot.columns]
    if len(protein_list) < min_p:
        have = set(protein_list)
        protein_list += [p for p in stats["protein"] if p not in have][:min_p - len(protein_list)]
    return prot[protein_list]

created = []

# V1 Ultra-Conservative: variance > 98.3 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.983)]["protein"].tolist())
fname = f"prot_1_ultra_conservative_{len(df.columns)}proteins.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V1","name":"ultra_conservative","n":len(df.columns),"logic":"Variance > 98.3 pct","file":fname})
print(f"  V1 ultra_conservative : {len(df.columns):,} proteins")

# V2 Conservative: variance > 97.5 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.975)]["protein"].tolist())
fname = f"prot_2_conservative_{len(df.columns)}proteins.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V2","name":"conservative","n":len(df.columns),"logic":"Variance > 97.5 pct","file":fname})
print(f"  V2 conservative       : {len(df.columns):,} proteins")

# V3 Standard: variance > 95.9 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.959)]["protein"].tolist())
fname = f"prot_3_standard_{len(df.columns)}proteins.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V3","name":"standard","n":len(df.columns),"logic":"Variance > 95.9 pct","file":fname})
print(f"  V3 standard           : {len(df.columns):,} proteins")

# V4 FDR-significant: top TARGET_BROAD by composite (FDR fallback expected)
fdr_proteins = stats[stats["fdr"] < 0.05].head(TARGET_BROAD)["protein"].tolist()
df    = build(fdr_proteins, min_p=TARGET_BROAD)
fname = f"prot_4_fdr_significant_{len(df.columns)}proteins.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V4","name":"fdr_significant","n":len(df.columns),"logic":f"FDR<0.05, top {TARGET_BROAD} composite fallback","file":fname})
print(f"  V4 fdr_significant    : {len(df.columns):,} proteins  (raw FDR<0.05: {int((stats['fdr']<0.05).sum())})")

# V5 Balanced: top 25% variance then top 10% Spearman
var_sub     = stats[stats["variance"] > stats["variance"].quantile(0.75)]
corr_thresh = var_sub["abs_spearman"].quantile(0.90)
df    = build(var_sub[var_sub["abs_spearman"] > corr_thresh]["protein"].tolist())
fname = f"prot_5_balanced_{len(df.columns)}proteins.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V5","name":"balanced","n":len(df.columns),"logic":"Var top 25% -> Spearman top 10%","file":fname})
print(f"  V5 balanced           : {len(df.columns):,} proteins")

# V6 Correlation-based: abs Spearman > 97.5 percentile
df    = build(stats[stats["abs_spearman"] > stats["abs_spearman"].quantile(0.975)]["protein"].tolist())
fname = f"prot_6_correlation_{len(df.columns)}proteins.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V6","name":"correlation","n":len(df.columns),"logic":"Abs Spearman > 97.5 pct","file":fname})
print(f"  V6 correlation        : {len(df.columns):,} proteins")

# V7 Top Correlated: top 100 pos + top 100 neg
# Note: proteome has ~460 proteins; 100+100 may overlap or exhaust the pool.
# build() pads to MIN_PROTEINS if needed.
top_pos = stats[stats["spearman_rho"] > 0].head(100)["protein"].tolist()
top_neg = stats[stats["spearman_rho"] < 0].tail(100)["protein"].tolist()
df      = build(sorted(set(top_pos + top_neg)))
stats[stats["protein"].isin(set(top_pos + top_neg))][
    ["protein","spearman_rho","pval","fdr","variance"]
].to_csv(OUTPUT_DIR / "prot_7_top_correlated_annotated.csv", index=False)
fname = f"prot_7_top_correlated_{len(df.columns)}proteins.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V7","name":"top_correlated","n":len(df.columns),"logic":"Top 100 pos + top 100 neg Spearman","file":fname})
print(f"  V7 top_correlated     : {len(df.columns):,} proteins")

# V8 Composite: top TARGET_BROAD by average rank
df    = build(stats.head(TARGET_BROAD)["protein"].tolist(), min_p=TARGET_BROAD)
fname = f"prot_8_composite_{len(df.columns)}proteins.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V8","name":"composite","n":len(df.columns),"logic":f"Avg rank Spearman+MI+Dcor, top {TARGET_BROAD}","file":fname})
print(f"  V8 composite          : {len(df.columns):,} proteins")

# Auxiliary files
outcome.to_csv(OUTPUT_DIR / "outcome.csv")
stats.to_csv(OUTPUT_DIR / "prot_statistics_complete.csv", index=False)
summary = pd.DataFrame(created)
summary.to_csv(OUTPUT_DIR / "datasets_summary.csv", index=False)

print()
print(summary[["V","name","n","logic"]].to_string(index=False))
print(f"Saved to: {OUTPUT_DIR}")


Creating 8 dataset variants...
Output: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Proteins\statistical_filtered
Note: proteome is small (463 proteins); TARGET_BROAD=200
  V1 ultra_conservative : 50 proteins
  V2 conservative       : 50 proteins
  V3 standard           : 50 proteins
  V4 fdr_significant    : 200 proteins  (raw FDR<0.05: 115)
  V5 balanced           : 50 proteins
  V6 correlation        : 50 proteins
  V7 top_correlated     : 200 proteins
  V8 composite          : 200 proteins

 V               name   n                                logic
V1 ultra_conservative  50                  Variance > 98.3 pct
V2       conservative  50                  Variance > 97.5 pct
V3           standard  50                  Variance > 95.9 pct
V4    fdr_significant 200 FDR<0.05, top 200 composite fallback
V5           balanced  50      Var top 25% -> Spearman top 10%
V6        correlation  50              Abs Spearman > 97.5 pct
V7     top_correlated 200   Top 100 pos + 